# `swift_too` module

## Async queueing pattern for Swift API jobs

### API version = 1.2, `swifttools` version 4.0

#### Author: Jamie A. Kennea (Penn State)

`QueryJob` is deprecated in this branch.
This notebook demonstrates the recommended async workflow that applies across API request classes that support queueing:

1. Build a request object.
2. Call `queue()` to submit asynchronously.
3. Poll the same object for completion.
4. Read results directly from that object.

`VisQuery` is used here only as a concrete example of the general pattern.

In [ ]:
from time import sleep

from swifttools.swift_too import VisQuery

Previously we would have defined here our username and shared_secret, however in `swifttools 2.2`, for most requests you can skip this and they will default to an anonymous user. For this notebook we'll use the anonymous login to make things simpler.

For this demo we queue a lightweight request so we can show async submission and polling in action. `VisQuery` is simply the example class used to illustrate the workflow.

In [ ]:
vq = VisQuery()
vq.name = "Crab"
vq.hires = True
vq.length = 1
if vq.queue():
    print(f"Queued visibility request. status={vq.status.status}")
else:
    print(f"Error: {vq.status.errors}")

Because we used `queue()`, execution returns immediately and processing may still be in progress. The next cells show how to poll until completion.



Now check progress by inspecting the same queued request object. This polling approach is the general async pattern used across queue-enabled API classes.

In [ ]:
print(f"Current status: {vq.status.status}; jobnumber={vq.status.jobnumber}")
qj = vq

You can pass key parameters as constructor arguments. In this notebook we set attributes explicitly first, then queue the request so status changes are easy to observe step by step.

In [ ]:
while not qj.complete:
    print(f"Current status: {qj.status.status}; waiting for completion...")
    sleep(1)

print(f"Final status: {qj.status.status}")

A queued request is complete when status reaches `Accepted`. If input validation or request handling fails, status may become `Rejected` and details will appear in `status.errors`.

The loop above polls once per second to avoid overloading the API.

Once accepted, inspect the returned object and its result fields.

In [ ]:
qj

The returned object can be displayed in table form. In this `VisQuery` example, the primary result field is `windows`. Other queue-enabled classes expose their own class-specific result fields.

In [ ]:
qj.windows

Other parameters that might be of interest are the `timestamp`, `began` and `completed` values, which allow you to find out when the job was submitted, when it started processing and when it finished. For example:

In [ ]:
print(f"Request status is {qj.status.status}; returned {len(qj.windows)} visibility windows.")

As usual, times reported back are UT. That is it! If all is correct, then the following show the upcoming or current *Swift* visibility window for Sgr A*!

In [ ]:
_ = [print(win) for win in qj]

This first returned window marks the earliest visibility start for the queried interval. In general, equivalent timing fields in other classes can be interpreted the same way for their domain-specific outputs.

In [ ]:
if len(qj) > 0 and qj.begin is not None:
    print(qj[0][0] - qj.begin)
else:
    print("No windows available.")

Note due to the weird way datetime displays negative timediffs, you might see something like `-1 day, 23:59:19` if the value is negative!